In [3]:
import os
import glob
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter

DB_NAME = "vector_db"
KNOWLEDGE_BASE =  "knowledge-base"

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

C:\Users\airtr\AppData\Local\Temp\ipykernel_19880\3984187573.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [4]:
def fetch_documents():
    folders = glob.glob(KNOWLEDGE_BASE + "/*")
    documents = []
    for folder in folders:
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(
            folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
        )
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)
    return documents

In [5]:
documents = fetch_documents()

In [6]:
# 이 부분에서 청크 사이즈를 잘 조절 해야 함
def create_chunks(documents):

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)

    chunks = text_splitter.split_documents(documents)
    return chunks

In [7]:
chunks = create_chunks(documents)

In [8]:
def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"벡터 저장소에 {dimensions:,}차원 벡터가 {count:,}개 있습니다")
    return vectorstore

In [10]:
vectorstore = create_embeddings(chunks)

벡터 저장소에 1,024차원 벡터가 543개 있습니다


In [11]:
retriever = vectorstore.as_retriever()

In [13]:
retriever.invoke("Alex Chen 연간 성과")

[Document(id='2d8e94e1-ae18-4093-8662-46f5758446d8', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Alex Chen.md'}, page_content='# 인사 기록\n\n# Alex Chen\n\n## 요약\n- **생년월일:** 1990년 3월 15일\n- **직책:** 백엔드 소프트웨어 엔지니어 (Backend Software Engineer)\n- **근무지:** 캘리포니아 주 샌프란시스코\n- **현재 연봉:** $115,000  \n\n## Insurellm 경력 진행\n- **2020년 4월:** Insurellm에 주니어 백엔드 개발자로 입사. 고객 데이터 보안을 강화하는 API 구축에 주력함.\n- **2021년 10월:** 백엔드 소프트웨어 엔지니어로 승진. 회사의 성장하는 플랫폼을 지원하기 위한 마이크로서비스 아키텍처 개발 핵심 프로젝트의 리더십을 맡음.\n- **2023년 3월:** 백엔드 서비스 확장에서 모범적인 성과를 인정받아 시니어 백엔드 소프트웨어 엔지니어 타이틀을 수여받음. 6개월간 다운타임을 30% 감소시킴.'),
 Document(id='d403128b-5865-4b90-84c4-9a5068243af8', metadata={'source': 'knowledge-base\\employees\\Alex Thomson.md', 'doc_type': 'employees'}, page_content='## 연간 성과 이력  \n- **2022년** - "기대치 초과(Exceeds Expectations)"로 평가됨. Alex Thomson은 입사 첫 3개월 만에 영업 목표의 150%를 달성함.  \n- **2023년** - "우수(Outstanding)"로 평가됨. 혁신적인 리드 발굴 전략으로 영업팀의 유효 리드를 30% 증가시킨 공로를 인정받음.  \n\n### 주요 성과:\n- 인바운드 리드에 대한 응답 시